In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
import os
import shutil
from collections import defaultdict
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
from tqdm import tqdm

from DefectParamsDistribution.DataBearingWithISO import DataBearingWithISO
from Lib_classes.DataFrameFilesBuilderStack import DataFrameStackBuilder
from Lib_classes.FileArchitecture import PdPickleFileArchitecture
from Lib_classes.Logger import Logger, LoggerEmpty
from Lib_classes.Paths import Paths
from Source.Source_aggregate.Bearing.Architectures.CWRUFileArchitectures import (
    CWRUFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.HitwzcFileArchitectures import (
    HitwzcFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.HUSTFileArchitectures import (
    HUSTFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.KAtFileArchitectures import (
    KAtFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.MFPTFileArchitectures import (
    MFPTFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.MultiDomainFileArchitectures import (
    MultiDomainFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.SCAFileArchitectures import (
    SCAFileArchitectures,
)
from Source.Source_aggregate.Bearing.Architectures.XJTU_SYFileArchitectures import (
    XJTU_SYFileArchitectures,
)
from Source.Source_aggregate.Callbacks.BearingParamsFillerCallback import (
    BearingParamsFillerCallback,
)
from Source.Source_aggregate.Callbacks.RpmFilter import RpmFilter
from Source.Source_aggregate.SourceAggregator import SourceAggregator
from Source.Source_aggregate.SourceAggregatorWithCallbacks import (
    SourceAggregatorWithCallbacks,
)
from Source.Utils.BalancedChunkRunner.SkLearnBalancedChunkRunner import (
    SkLearnBalancedChunkRunner,
)
from Source.Utils.GroupIntegrityEvaluator import GroupIntegrityEvaluator
from Source.Utils.MultiLabelStratificationError import MultiLabelStratificationError
from Source.Utils.Signature.ASignature import ASignature
from Source.Utils.Signature.BearingFaultSignatureMulti import BearingFaultSignatureMulti
from Source.Utils.Signature.BearingPathSignatureMulti import BearingPathSignatureMulti
from Source.Utils.Signature.MultiSignature import MultiSignature
from Source.Utils.Signature.MultiSimpleStraficater import MultiSimpleStraficater
from Source.Utils.Signature.PathSignature import PathSignature
from Source.Utils.Spliters.EqualSizeForScikitSplitter import EqualSizeForScikitSplitter
from Source.Utils.Spliters.MultilabelStratifiedGroupKFold import (
    MultilabelStratifiedGroupKFold,
)


In [40]:
target_folder = os.path.join(Paths.dir_dataframe_pickles, 'Bearing')

In [41]:
bind_classess = {
    'CWRU': CWRUFileArchitectures,
    # 'fm6xzxnf36-2': FM6FileArchitectures,
    'HUST': HUSTFileArchitectures,
    # 'JNU': JNUFileArchitectures,
    'KAt': KAtFileArchitectures,
    'MFPT': MFPTFileArchitectures,
    'Multi-domain': MultiDomainFileArchitectures,
    'SCA': SCAFileArchitectures,
    'hitwzc': HitwzcFileArchitectures,
    'XJTU-SY':XJTU_SYFileArchitectures,
}

In [42]:
rf = RpmFilter()
bear_params = BearingParamsFillerCallback(
    need_params = ['zT', 'dT', 'd0', 'rotation', 'rolling_body'],
    bearing_config = DataBearingWithISO(),
    logger_warn = LoggerEmpty('BearingParamsFiller(warn)'),
    logger_error = Logger('BearingParamsFiller(error)')
)

In [ ]:
sa = SourceAggregatorWithCallbacks(
    max_stack_size=1000, 
    path = os.path.join(Paths.dir_dataframe_pickles,'Bearing'),
    bind_classess = bind_classess,
    callbacks=[rf, bear_params], 
)

In [44]:
sa()

XJTU_SYFileArchitectures/files: 100%|██████████| 287/287 [00:07<00:00, 36.40it/s]


In [45]:
set(bear_params.undefined_bearing)

set()

In [46]:
len(rf.filtered_rows)

154

In [47]:
files = defaultdict(int)
for row in rf.filtered_rows:
    files[row['original_path']]+=1
    
pprint(files)

defaultdict(<class 'int'>,
            {'d:\\defect\\vibration-sources\\data\\external\\source\\SCA\\11\\train.mat': 4,
             'd:\\defect\\vibration-sources\\data\\external\\source\\SCA\\1\\test.mat': 16,
             'd:\\defect\\vibration-sources\\data\\external\\source\\SCA\\1\\train.mat': 10,
             'd:\\defect\\vibration-sources\\data\\external\\source\\SCA\\5\\train.mat': 48,
             'd:\\defect\\vibration-sources\\data\\external\\source\\SCA\\6\\test.mat': 8,
             'd:\\defect\\vibration-sources\\data\\external\\source\\SCA\\6\\train.mat': 68})


### BalancingSplit

In [48]:
n_chunks = 5
rng = np.random.default_rng(0)

In [49]:
# class BearingFaultSignatureMulti(BearingFaultSignature, MultiSignature):
#     """
#     Concrete strategy for Bearing datasets. 
#     Combines 'original_path' source and fault labels into a unique key.
#     """
#     def __init__(self, class_columns: List[str], sources:list):
#         self.class_columns =  class_columns
#         self._path_labels = dict(zip(sources, range(len(sources))))

#     def get_signature(self, row: pd.Series) -> np.ndarray:
#         path = self._path_labels[self._get_path(row)]
#         fault_vector:np.ndarray = row[self.class_columns].values.astype(np.float32) # type: ignore
#         return fault_vector + path*2
    
#     def get_size(self):
#         return len(self.class_columns)

In [50]:
all_sources = list(filter(lambda x: x.find('.') == -1, os.listdir(Paths.dir_source)))
all_sources

['CWRU',
 'fm6xzxnf36-2',
 'HUST',
 'JNU',
 'KAt',
 'MFPT',
 'Multi-domain',
 'Multi-domain2',
 'Multi-domain3',
 'SCA',
 'Self-built dataset',
 'SpectraQuest MFS dataset',
 'XJTU-SY']

In [51]:
pfa = PdPickleFileArchitecture.open(target_folder)
bcr = SkLearnBalancedChunkRunner(
    pfa,
    EqualSizeForScikitSplitter(MultilabelStratifiedGroupKFold(n_splits = n_chunks), rng, is_equalizing = False, add_remainder = False),
    BearingFaultSignatureMulti(SourceAggregator.REAL_CLASS_ORDER, all_sources),
    PathSignature(),
    rng,
)

In [52]:
len(pfa.rows())

11042

In [53]:
chunks = bcr.generate_chunk_indices()

In [54]:
count_chunks = {k: len(v) for k,v in chunks.items()}
count_chunks

{0: 2506, 1: 2265, 2: 1979, 3: 2013, 4: 2279}

In [55]:
balance_chunk_path = os.path.join(target_folder, SourceAggregator.CHUNK_FOLDER_NAME)

In [56]:
pmf = PdPickleFileArchitecture(
    balance_chunk_path,
    max_stack_size=len(next(iter(chunks.values())))
)
for k_, df in tqdm(bcr.provide_chunks(chunks), total=n_chunks):
    pmf.add_file(df)


100%|██████████| 5/5 [01:15<00:00, 15.12s/it]


In [57]:
for file in os.listdir(os.path.join(target_folder)):
    path = os.path.join(target_folder, file)
    if not os.path.isdir(path) and path.endswith(PdPickleFileArchitecture.extension):
        os.remove(os.path.abspath(path))

## Recheck balances

In [58]:
class FilesSegregationGroups(ASignature):    
    def get_signature(self, row: pd.Series) -> int:
        """Returns a unique string key representing the stratum of the row."""
        return row['original_path']

In [59]:
class_error = MultiLabelStratificationError(MultiSimpleStraficater(SourceAggregator.REAL_CLASS_ORDER))
dataset_error = MultiLabelStratificationError(BearingPathSignatureMulti(all_sources))
overvall_error = MultiLabelStratificationError(BearingFaultSignatureMulti(SourceAggregator.REAL_CLASS_ORDER, all_sources))

group_completeness = GroupIntegrityEvaluator(FilesSegregationGroups())

pdpickle = PdPickleFileArchitecture.open(balance_chunk_path, 5000)
for i in pdpickle.iter_stack_index():
    df = pdpickle.get_file(i)
    class_error.add_chunk(df)
    dataset_error.add_chunk(df)
    overvall_error.add_chunk(df)
    group_completeness.add_chunk(df, i)
    df = df[~(df['rpm'].map(lambda x: x < 3000 and x > 0))]
    if len(df) > 0:
        print("------------------------------")
        print(df)
        print("------------------------------")

[407  87 483]
[  85.    0.   21.    0.  496.    5.   76.   77.   76. 1353.    4.    4.
  309.]
[ 20.   0.   9.   0. 208.   2.  19.  19.  19.   0.   2.   1. 108.  19.
   0.   8.   0.   0.   0.  20.  19.  19.   0.   1.   1.   0.  40.   0.
   9.   0. 240.   2.  19.  19.  19.   1.   1.   2. 131.]
[526  85 577]
[  79.    0.   20.    0.  496.    3.   77.   77.   76. 1116.    4.    5.
  312.]
[ 21.   0.   9.   0. 208.   1.  19.  19.  19. 117.   2.   2. 109.  19.
   0.   8.   0.   0.   0.  19.  19.  19.   0.   0.   1.   0.  39.   0.
   9.   0. 240.   2.  19.  20.  19.  94.   2.   2. 131.]
[426 110 622]
[ 86.   0.  20.   0. 496.   4.  77.  76.  77. 826.   3.   3. 311.]
[ 20.   0.   8.   0. 208.   2.  19.  19.  20.  18.   1.   2. 109.  22.
   0.   8.   0.   0.   0.  19.  19.  19.  23.   0.   0.   0.  42.   0.
   9.   0. 240.   2.  19.  19.  19. 138.   2.   1. 131.]
[450  86 502]
[ 84.   0.  19.   0. 496.   4.  78.  76.  76. 860.   5.   4. 311.]
[ 22.   0.   8.   0. 208.   1.  20.  19.  19.  41. 

In [60]:
class_error.report()


--- Stratification Error Report ---
Overall Distribution Error: 10.27%

Error by Class:
inner_race           | Total samples: 2387   | Error: 12.50%
ball                 | Total samples: 455    | Error: 8.35%
outer_race           | Total samples: 2666   | Error: 9.95%


In [61]:
dataset_error.report()


--- Stratification Error Report ---
Overall Distribution Error: 4.99%

Error by Class:
CWRU                 | Total samples: 416    | Error: 2.60%
fm6xzxnf36-2         | Total samples: 0      | Error: 0.00%
HUST                 | Total samples: 99     | Error: 3.23%
JNU                  | Total samples: 0      | Error: 0.00%
KAt                  | Total samples: 2479   | Error: 0.06%
MFPT                 | Total samples: 20     | Error: 10.00%
Multi-domain         | Total samples: 384    | Error: 0.83%
Multi-domain2        | Total samples: 384    | Error: 0.83%
Multi-domain3        | Total samples: 381    | Error: 0.42%
SCA                  | Total samples: 5282   | Error: 16.16%
Self-built dataset   | Total samples: 21     | Error: 15.24%
SpectraQuest MFS dataset | Total samples: 21     | Error: 15.24%
XJTU-SY              | Total samples: 1555   | Error: 0.26%


In [62]:
overvall_error.report()


--- Stratification Error Report ---
Overall Distribution Error: 16.67%

Error by Class:
inner_race,CWRU      | Total samples: 103    | Error: 3.50%
inner_race,fm6xzxnf36-2 | Total samples: 0      | Error: 0.00%
inner_race,HUST      | Total samples: 42     | Error: 5.71%
inner_race,JNU       | Total samples: 0      | Error: 0.00%
inner_race,KAt       | Total samples: 1040   | Error: 0.00%
inner_race,MFPT      | Total samples: 7      | Error: 34.29%
inner_race,Multi-domain | Total samples: 96     | Error: 1.67%
inner_race,Multi-domain2 | Total samples: 96     | Error: 1.67%
inner_race,Multi-domain3 | Total samples: 96     | Error: 1.67%
inner_race,SCA       | Total samples: 346    | Error: 85.90%
inner_race,Self-built dataset | Total samples: 9      | Error: 17.78%
inner_race,SpectraQuest MFS dataset | Total samples: 9      | Error: 17.78%
inner_race,XJTU-SY   | Total samples: 543    | Error: 0.44%
ball,CWRU            | Total samples: 100    | Error: 4.00%
ball,fm6xzxnf36-2    | Total 

In [63]:
group_completeness.report()


--- Group Integrity & Leakage Report ---
Completeness Score: 1.0000
Total Unique Groups: 5525
Leaked Groups (Spanning multiple chunks): 0

No leakage.


## Test core

In [64]:
n = 1000
outputs = 3
features = 10
groups = 20

groups = np.random.randint(0, groups, n)
y = np.random.randint(0, 2, (n, outputs))
x = np.arange(n*features).reshape(n, features)

itr_straf = MultilabelStratifiedGroupKFold(n_splits = n_chunks)
chanks = []
for _, chunk in itr_straf.split(x, y, groups):
    chanks.append(chunk)


In [65]:
class SimpleExtractor(MultiSignature, ASignature):
    def __init__(self, columns):
        self.columns = columns
        self._labels = [str(i) for i in columns]
    def get_signature(self, df):
        if len(self.columns) != 1:
            return df[self.columns]
        else:
            return df[self.columns].to_numpy().squeeze().tolist()
    def get_size(self):
        return len(self.columns)

In [66]:
output_error = MultiLabelStratificationError(SimpleExtractor(list(range(outputs))))
groups_error = GroupIntegrityEvaluator(SimpleExtractor([0]))

In [67]:
for i, cha in enumerate(chanks):
    output_error.add_chunk(pd.DataFrame(y[cha]))
    groups_error.add_chunk(pd.DataFrame(groups[cha]), i)

[ 96 104 106]
[102 104 112]
[95 92 90]
[100 101 111]
[ 91  95 102]


In [68]:
output_error.report()


--- Stratification Error Report ---
Overall Distribution Error: 4.79%

Error by Class:
0                    | Total samples: 484    | Error: 3.47%
1                    | Total samples: 496    | Error: 4.60%
2                    | Total samples: 521    | Error: 6.30%


In [69]:
groups_error.report()


--- Group Integrity & Leakage Report ---
Completeness Score: 1.0000
Total Unique Groups: 20
Leaked Groups (Spanning multiple chunks): 0

No leakage.


## Make need file-structure

In [70]:
proportion = [3, 1, 1]
name = ['train', 'validate', 'test']

count = 0
for chunks, folder_name in zip(proportion, name):
    current_pdpickle = PdPickleFileArchitecture(os.path.join(target_folder, folder_name), len(pdpickle.get_file(count)))
    for i in range(chunks):
        df = pdpickle.get_file(count)
        current_pdpickle.add_file(df)
        count += 1

In [71]:
shutil.rmtree(balance_chunk_path)

In [72]:
SourceAggregatorWithCallbacks.split_groups(
    PdPickleFileArchitecture.open(os.path.join(target_folder, 'validate')),
)

Processing Source Files: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


In [73]:
train_path = os.path.join(target_folder,'train')
temp_train_path = os.path.join(target_folder,'train_temp')

train = PdPickleFileArchitecture.open(train_path)
with DataFrameStackBuilder(train.get_columns().to_list(), PdPickleFileArchitecture(temp_train_path, 1000)) as builder:
    for i in train.iter_stack_index():
        df = train.get_file(i)
        builder.add_df(df)
        
shutil.rmtree(train_path)
os.rename(temp_train_path, train_path)

In [74]:
SourceAggregatorWithCallbacks.split_groups(
    PdPickleFileArchitecture.open(os.path.join(target_folder, 'train')),
    output_base = Path(os.path.join(target_folder,'splits_train')),
)

Processing Source Files: 100%|██████████| 7/7 [00:06<00:00,  1.06it/s]
